In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Phase 1 - Target & Leakage Boundary

In [7]:
RANDOM_STATE = 123
DATA_DIR = '../02_data_cleaning/data/clean'

In [8]:
orders = pd.read_csv(f"{DATA_DIR}/Olist_Orders.csv",
                      parse_dates=["order_purchase_timestamp", "order_approved_at",
                                   "order_delivered_carrier_date",
                                   "order_delivered_customer_date",
                                   "order_estimated_delivery_date"])
items = pd.read_csv(f"{DATA_DIR}/Olist_Order_Items.csv")
customers = pd.read_csv(f"{DATA_DIR}/Olist_Customers.csv")
sellers = pd.read_csv(f"{DATA_DIR}/Olist_Sellers.csv")
products = pd.read_csv(f"{DATA_DIR}/Olist_Products.csv")
payments = pd.read_csv(f"{DATA_DIR}/Olist_Order_Payments.csv")

orders = orders[orders["order_status"] == "delivered"].copy()

In [11]:
def clean_unnamed(df):
    return df.loc[:, ~df.columns.str.contains("^Unnamed")]

orders = clean_unnamed(orders)
items = clean_unnamed(items)
customers = clean_unnamed(customers)
sellers = clean_unnamed(sellers)
products = clean_unnamed(products)
payments = clean_unnamed(payments)

In [12]:
df = (orders
      .merge(items, on="order_id", how="left")
      .merge(customers, on="customer_id", how="left")
      .merge(sellers, on="seller_id", how="left")
      .merge(products, on="product_id", how="left")
      .merge(payments, on="order_id", how="left"))

In [13]:
df

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,was_delivered,order_item_id,...,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,payment_sequential,payment_type,payment_installments,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,1,...,268.0,4.0,500.0,19.0,8.0,13.0,1.0,credit_card,1.0,18.12
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,1,...,268.0,4.0,500.0,19.0,8.0,13.0,3.0,voucher,1.0,2.00
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,1,...,268.0,4.0,500.0,19.0,8.0,13.0,2.0,voucher,1.0,18.59
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,True,1,...,178.0,1.0,400.0,19.0,13.0,19.0,1.0,boleto,1.0,141.46
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,True,1,...,232.0,1.0,420.0,24.0,19.0,21.0,1.0,credit_card,3.0,179.12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115033,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,True,1,...,828.0,4.0,4950.0,40.0,10.0,40.0,1.0,credit_card,3.0,195.00
115034,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,True,1,...,500.0,2.0,13300.0,32.0,90.0,22.0,1.0,credit_card,5.0,271.01
115035,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,True,1,...,1893.0,1.0,6550.0,20.0,20.0,20.0,1.0,credit_card,4.0,441.16
115036,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,True,2,...,1893.0,1.0,6550.0,20.0,20.0,20.0,1.0,credit_card,4.0,441.16


In [14]:
df['is_late'] = (df['order_delivered_customer_date'] > df['order_estimated_delivery_date']).astype(int)

In [19]:
raw_col_count = df.shape[1]
raw_col_count

35

In [16]:
print(df.shape, df['is_late'].value_counts(normalize=True))

(115038, 35) is_late
0    0.921539
1    0.078461
Name: proportion, dtype: float64


# Phase 2 - Time-Based Split

In [20]:
df = df.sort_values('order_purchase_timestamp')
idx = int(len(df) * 0.8)
date = df.iloc[idx]['order_purchase_timestamp']

In [21]:
train = df[df['order_purchase_timestamp'] < date].copy()
test = df[df['order_purchase_timestamp'] >= date].copy()

In [22]:
print(train.shape, test.shape)

(92030, 35) (23008, 35)


# Phase 3 - Feature Creation

In [23]:
for split in (train, test):
    ts = split['order_purchase_timestamp']
    split['purchase_weekday'] = ts.dt.dayofweek
    split['purchase_month'] = ts.dt.month
    split['purchase_hour'] = ts.dt.hour
    split['is_weekend'] = split['purchase_weekday'].isin([5, 6]).astype(int)

    split["freight_to_price_ratio"] = split["freight_value"] / split["price"].replace(0, np.nan)
    split["item_count"] = split.groupby("order_id")["order_item_id"].transform("count")
    split["product_weight_g"] = split["product_weight_g"].fillna(split["product_weight_g"].median())
    split["product_volume_cm3"] = (split["product_length_cm"].fillna(0) * split["product_height_cm"].fillna(0) * split["product_width_cm"].fillna(0))

    split["same_state"] = (split["seller_state"] == split["customer_state"]).astype(int)

In [24]:
seller_late_rate = train.groupby("seller_id")["is_late"].mean()
global_late_rate = train["is_late"].mean()
category_avg_freight = train.groupby("product_category_name")["freight_value"].mean()
global_avg_freight = train["freight_value"].mean()

In [25]:
for split in (train, test):
    split["seller_late_rate_hist"] = split["seller_id"].map(seller_late_rate).fillna(global_late_rate)
    split["category_avg_freight_hist"] = (split["product_category_name"].map(category_avg_freight).fillna(global_avg_freight))

In [ ]:
created_feature_count = len(['purchase_weekday', 'purchase'])